In [3]:
import pandas as pd

df = pd.read_csv('processed_nlp_data.csv')

df.head()

,Customer Age,Customer Gender,Product Purchased,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,...,purchase_day,desc_length,subject_length,cleaned_description,cleaned_subject,final_description,final_subject,lemmatized_description,lemmatized_subject,final_text
0,32,Other,GoPro Hero,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,...,22,284,13,im having an issue with the productpurchased p...,product setup,im issue productpurchased please assist billin...,product setup,im issue productpurchased please assist billin...,product setup,product setup im issue productpurchased please...
1,42,Female,LG Smart TV,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,...,22,282,24,im having an issue with the productpurchased p...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,im issue productpurchased please assist need c...,peripheral compatibility,peripheral compatibility im issue productpurch...
2,48,Other,Dell XPS,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,...,14,275,15,im facing a problem with my productpurchased t...,network problem,im facing problem productpurchased productpurc...,network problem,im facing problem productpurchased productpurc...,network problem,network problem im facing problem productpurch...
3,27,Female,Microsoft Office,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,...,13,262,14,im having an issue with the productpurchased p...,account access,im issue productpurchased please assist proble...,account access,im issue productpurchased please assist proble...,account access,account access im issue productpurchased pleas...
4,67,Female,Autodesk AutoCAD,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,...,4,333,9,im having an issue with the productpurchased p...,data loss,im issue productpurchased please assist note s...,data loss,im issue productpurchased please assist note s...,data loss,data loss im issue productpurchased please ass...


In [4]:
# Create better text feature
df['better_text'] = df['Ticket Subject'] + " " + df['Product Purchased'].fillna('')

In [5]:
# Features and target
X = df['better_text']
y = df['Ticket Priority']

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [9]:
!pip install xgboost

In [10]:
from xgboost import XGBClassifier

In [11]:
model = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

model.fit(X_train_tfidf, y_train_enc)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [17:12:37] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [12]:
y_pred = model.predict(X_test_tfidf)

In [13]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test_enc, y_pred))

Accuracy: 0.23140495867768596
